<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/Chapter_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 9: Algoritma Multiclass dan Multilabel

<div class="alert alert-info">
<b>Ringkasan Bab:</b> Bab ini mengeksplorasi strategi untuk menangani masalah klasifikasi yang lebih kompleks. Kita akan mempelajari perbedaan antara klasifikasi Multiclass (memilih satu dari banyak) dan Multilabel (memilih banyak dari banyak), serta menggunakan meta-estimator seperti OneVsRestClassifier.
</div>

## 1. Strategi Multiclass: One-vs-Rest (OvR)
Banyak algoritma klasifikasi secara alami hanya mendukung dua kelas (biner). Untuk menangani banyak kelas, scikit-learn menggunakan strategi **One-vs-Rest**. Jika kita punya kelas A, B, dan C, model akan membuat 3 pengklasifikasi biner:
1. A vs [B, C]
2. B vs [A, C]
3. C vs [A, B]

In [25]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load dataset Iris (3 kelas)
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Menggunakan SVC (yang aslinya biner) dalam skema One-vs-Rest
ovr_clf = OneVsRestClassifier(SVC(kernel='linear', probability=True))
ovr_clf.fit(X_train, y_train)

print(f"Akurasi Multiclass OvR: {ovr_clf.score(X_test, y_test) * 100:.2f}%")
print(f"Jumlah estimator (model biner) yang dibuat: {len(ovr_clf.estimators_)}")

Akurasi Multiclass OvR: 95.56%
Jumlah estimator (model biner) yang dibuat: 3


---
## 2. Klasifikasi Multilabel
Dalam klasifikasi **Multilabel**, setiap contoh data dapat memiliki lebih dari satu label target. Contohnya: sebuah film bisa memiliki genre "Action" sekaligus "Sci-Fi".



Kita menggunakan `MultiLabelBinarizer` untuk mengubah daftar label menjadi format matriks biner yang bisa diproses oleh model.

In [26]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# Contoh data: 3 dokumen dengan label genre berbeda-masing
# Dokumen 0: Action, Sci-Fi
# Dokumen 1: Drama
# Dokumen 2: Action, Drama
y_genres = [(['Action', 'Sci-Fi']), (['Drama']), (['Action', 'Drama'])]

# Mengubah label teks menjadi matriks biner
mlb = MultiLabelBinarizer()
y_binarized = mlb.fit_transform(y_genres)

print("Nama Kelas:", mlb.classes_)
print("Matriks Label Biner:\n", y_binarized)

# Contoh fitur sederhana (dummy)
X_dummy = np.array([[1, 2], [3, 4], [1, 5]])

# Melatih model Multilabel
multilabel_clf = OneVsRestClassifier(DecisionTreeClassifier())
multilabel_clf.fit(X_dummy, y_binarized)

print("\nBerhasil melatih model Multilabel.")

Nama Kelas: ['Action' 'Drama' 'Sci-Fi']
Matriks Label Biner:
 [[1 0 1]
 [0 1 0]
 [1 1 0]]

Berhasil melatih model Multilabel.


---
## 3. Output Code Classifier (Output Error-Correcting)
Selain OvR, ada teknik lain yang disebut **Output Code Classifier**. Teknik ini memetakan setiap kelas ke dalam kode biner unik. Ini berguna untuk meningkatkan ketahanan (robustness) klasifikasi jika salah satu model biner gagal.

In [27]:
from sklearn.multiclass import OutputCodeClassifier

# Menggunakan strategi Output Code dengan SVC
occ_clf = OutputCodeClassifier(SVC(kernel='linear'), code_size=2, random_state=42)
occ_clf.fit(X_train, y_train)

print(f"Akurasi Output Code Classifier: {occ_clf.score(X_test, y_test) * 100:.2f}%")

Akurasi Output Code Classifier: 100.00%
